In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import os

def load_datasets(path_697: str, path_320: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load both UCI datasets from given file paths."""
    df697 = pd.read_csv(path_697, sep=";")
    df320 = pd.read_csv(path_320, sep=";")
    print(f"[697] Loaded {df697.shape[0]} rows, {df697.shape[1]} columns")
    print(f"[320] Loaded {df320.shape[0]} rows, {df320.shape[1]} columns")
    return df697, df320

def engineer_697(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()

    grade_cols = [c for c in df.columns if "Curricular units" in c and " grade" in c.lower()]
    if grade_cols:
        raw_avg = df[grade_cols].mean(axis=1)
        out["previous_grade_score"] = (raw_avg/20*100).clip(0,100)

    else:
        out["previous_grade_score"]=np.nan

    if "Admission grade " in df.columns:
        out["background_academic_score"]=(df["Admission grade"]/200*100).clip(0,100)
    
    else :
        out["background_academic_score"]=np.nan

    enroll_cols = [c for c in df.columns if "Curricular units" in c and "enrolled" in c.lower()]
    if enroll_cols:
        total = df[enroll_cols].sum(axis=1)
        out["enrolled_units_count"]=total.clip(3,14)

    else:
        out["enrolled_units_count"]=np.nan

    fails_cols = [c for c in df.columns if "without evaluations" in c.lower()]
    if fails_cols:
        out["past_failures_count"]=df[fails_cols].sum(axis=1).clip(0,10)
    else:
        out["past_failures_count"]=np.nan
    
    approved_cols=[c for c in df.columns if "Curricular units" in c and "approved"in c.lower()]
    if approved_cols:
        out["approved_units_count"]=df[approved_cols].sum(axis=1).clip(0,30)
    else:
        out["approved_units_count"]=np.nan
    
    out["study_time_level_raw"] = 5.0
    out["absence_rate_raw"]=1
    
    if "Age at enrollment" in df.columns:
        out["age"]=df["Age at enrollment"].clip(17,60)
    else:
        out["age"]=np.nan
    
    edu_cols=[c for c in df.columns if "education" in c.lower() and ("mother" in c.lower() or "father" in c.lower())]
    if len(edu_cols) >=1:
        out["parent_education_raw"]=df[edu_cols].mean(axis=1)
    else:
        out["parent_education_raw"]=np.nan

    if "Target" in df.columns:
        out["student_status"]=df["Target"]
    else:
        out["student_status"]=np.nan
    
    out["source"]="697"
    return out

def engineer_320(df:pd.DataFrame) ->pd.DataFrame:
    out = pd.DataFrame()
    if "G1" in df.columns and "G2" in df.columns:
        out["previous_grade_score"]=((df["G1"]+df["G2"])/2/20*100).clip(0,100)
    else:
        out["previous_grade_score"]=np.nan
    
    out["background_academic_score"]=np.nan

    np.random.seed(42)
    out["enrolled_units_count"]=np.random.randint(3,8,size=len(df)).astype(float)
    
    if "failures" in df.columns:
        out["past_failures_count"] = df["failures"].clip(0,10)
    else:
        out["past_failures_count"]=0

    out["approved_units_count"]=np.nan
    if "studytime" in df.columns:
         # Map: 1→2.5h, 2→3.5h, 3→7h, 4→12h
        study_map = {1: 2.5, 2: 3.5 , 3: 7.0, 4: 12.0}
        out["study_time_level_raw"] = df["studytime"].map(study_map).fillna(3.5)
    else:
        out["study_time_level_raw"]=3.5
    
    if "absences" in df.columns:
        def map_absence(x):
            if x == 0:
                return 0
            elif x <= 10:
                return 1
            else:
                return 2
        
        out["absence_rate_raw"] = df["absences"].apply(map_absence)

    else:
        out["absence_rate_raw"] = 1
    
    if "age" in df.columns:
        out["age"] = df["age"].clip(15,30)
    else:
        out["age"]=np.nan

    edu_cols = [c for c in df.columns if c in ["Medu", "Fedu"]]
    if len(edu_cols) >= 1:
        out["parent_education_raw"] = df[edu_cols].mean(axis=1)
    else:
        out["parent_education_raw"]=np.nan

    if "G3" in df.columns:
        def derive_status(g3):
            if g3<10:
                return "Dropout"
            elif g3<15:
                return "Enrolled"
            else:
                return "Graduate"

        out["student_status"]= df["G3"].apply(derive_status)
    else:
        out["student_status"] = np.nan
    
    out["source"] = "320"
    return out

def merge_and_preprocess(df697_eng:pd.DataFrame,df320_eng:pd.DataFrame) ->pd.DataFrame:
    merged = pd.concat([df697_eng,df320_eng],ignore_index=True)
    print(f"\[Merge] Combined shape: {merged.shape}")
    
   
    numeric_cols = [
        "previous_grade_score", "background_academic_score",
        "enrolled_units_count", "past_failures_count",
        "approved_units_count", "study_time_level_raw",
        "age", "parent_education_raw"
    ]

    for col in numeric_cols:
        median_val = merged[col].median()
        n_missing = merged[col].isna().sum()
        merged[col] = merged[col].fillna(median_val)

        if n_missing > 0:
            print(f"Imputed {n_missing} missing values in '{col}' with median = {median_val:.2f}")

    merged["parent_education_level_norm"] = (merged["parent_education_raw"]/6*100).clip(0,100)

    def categorize_difficulty(n):
        if n<=4:
            return "low"
        elif n<=6:
            return "Medium"
        else:
            return "High"

    merged["difficulty_level"]=merged["enrolled_units_count"].apply(categorize_difficulty)

    def categorize_study(h):
        if h<5:
            return "Low"
        elif h<=10:
            return "Medium"
        else:
            return "High"

    merged["study_time_level"] = merged["study_time_level_raw"].apply(categorize_study)
    absence_map = {0:"Rare" , 1: "Sometimes", 2: "Often"}
    merged["absence_rate"] = merged["absence_rate_raw"].map(absence_map).fillna("Sometimes")

    def categorize_age(a):
        if a<=20:
            return "18-20"
        elif a <=23:
            return "21-23"
        else:
            return "24+"

    merged["age_group"] = merged["age"].apply(categorize_age)
    label_encoders = {}
    cat_cols = ["difficulty_level", "study_time_level","absence_rate","age_group"]
    for col in cat_cols:
        le=LabelEncoder()
        merged[col + "_enc"] = le.fit_transform(merged[col])
        label_encoders[col]=le
        print(f"Encoded '{col}': {dict(zip(le.classes_,le.transform(le.classes_)))}")

    merged["student_status"] = merged["student_status"].str.strip()
    target_le = LabelEncoder()
    merged["student_status_enc"]=target_le.fit_transform(merged["student_status"].fillna("Enrolled"))
    print(f" Target classes: {dict(zip(target_le.classes_, target_le.transform(target_le.classes_)))}")
    return merged, label_encoders , target_le

def build_final_df(merged: pd.DataFrame) ->pd.DataFrame:
    feature_cols = [
        "previous_grade_score",
        "background_academic_score",
        "enrolled_units_count",
        "difficulty_level_enc",
        "past_failures_count",
        "approved_units_count",
        "study_time_level_enc",
        "absence_rate_enc",
        "age",
        "parent_education_level_norm",
        "student_status_enc",
        "source"
    ]
    final = merged[feature_cols].copy()
    final = final.rename(columns={
        "difficulty_level_enc": "difficulty_level",
        "study_time_level_enc": "study_time_level",
        "absence_rate_enc": "absence_rate",
        "parent_education_level_norm": "parent_education_level",
        "student_status_enc": "student_status"
    })
    print(f"\n[Final] Training dataframe shape: {final.shape}")
    print(f"[Final] Missing values:\n{final.isnull().sum()}")
    print(f"\n[Final] Class distribution:\n{final['student_status'].value_counts()}")
    print(f"\n[Final] Feature dtypes:\n{final.dtypes}")
    print(f"\n[Final] Sample rows:\n{final.head()}")
    return final


if __name__ == "__main__":
    # ── Update these paths to wherever you saved the UCI dataset files ──
    PATH_697 = "data/dataset_697.csv"   # semicolon-separated
    PATH_320 = "data/student-mat.csv"   # semicolon-separated (Math dataset)

    if not os.path.exists(PATH_697) or not os.path.exists(PATH_320):
        print("=" * 60)
        print("ERROR: Dataset files not found.")
        print(f"  Expected path 1: {PATH_697}")
        print(f"  Expected path 2: {PATH_320}")
        print("\nDownload instructions:")
        print("  Dataset 697: https://archive.ics.uci.edu/dataset/697/")
        print("  Dataset 320: https://archive.ics.uci.edu/dataset/320/")
        print("\nAfter downloading, update PATH_697 and PATH_320 at the")
        print("bottom of this file and run again.")
        print("=" * 60)
    else:
        df697_raw, df320_raw = load_datasets(PATH_697, PATH_320)

        print("\n── Engineering Dataset 697 features ──")
        df697_eng = engineer_697(df697_raw)

        print("\n── Engineering Dataset 320 features ──")
        df320_eng = engineer_320(df320_raw)

        print("\n── Merging and preprocessing ──")
        merged, label_encoders, target_le = merge_and_preprocess(df697_eng, df320_eng)

        print("\n── Building final training dataframe ──")
        final_df = build_final_df(merged)

        # Save outputs
        os.makedirs("output", exist_ok=True)
        final_df.to_csv("output/training_data.csv", index=False)
        print("\n✅ Saved to output/training_data.csv")

        # Also save label encoder mappings for reference
        import json
        mappings = {
            col: dict(zip(le.classes_.tolist(), le.transform(le.classes_).tolist()))
            for col, le in label_encoders.items()
        }
        mappings["student_status"] = dict(zip(
            target_le.classes_.tolist(),
            target_le.transform(target_le.classes_).tolist()
        ))
        with open("output/label_encodings.json", "w") as f:
            json.dump(mappings, f, indent=2)
        print("✅ Saved label encodings to output/label_encodings.json")

    








    



ERROR: Dataset files not found.
  Expected path 1: data/dataset_697.csv
  Expected path 2: data/student-mat.csv

Download instructions:
  Dataset 697: https://archive.ics.uci.edu/dataset/697/
  Dataset 320: https://archive.ics.uci.edu/dataset/320/

After downloading, update PATH_697 and PATH_320 at the
bottom of this file and run again.


<>:141: SyntaxWarning: invalid escape sequence '\['
<>:141: SyntaxWarning: invalid escape sequence '\['
C:\Users\yassi\AppData\Local\Temp\ipykernel_42540\3324518749.py:141: SyntaxWarning: invalid escape sequence '\['
  print(f"\[Merge] Combined shape: {merged.shape}")
